## Import ##

In [ ]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
# Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
import torch.nn as nn

class VideoClassifierLSTM(nn.Module):
    def __init__(self, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.fc = nn.Linear(self.dino_model.embed_dim, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):        
        dino_feature = self.dino_model(x)
        output = self.dropout(dino_feature)
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
video_lassifier_model = VideoClassifierLSTM(num_classes=744)

video_lassifier_model.load_state_dict(torch.load("/media/osero/SamsungSSD/CMPE_SSD/Pth_Files/FINE_TUNED_LEFT_MODEL2024-12-30_08-17-42_1.pth"))

model = video_lassifier_model.dino_model
model.to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Functions ##

In [ ]:
# import cv2
# import os
# import numpy as np

# # Step 2: Function to Extract Frames from Video
# def extract_frames(video_path):
#     """Extract frames from a video at a specified frame rate."""
#     cap = cv2.VideoCapture(video_path)

#     # Check if video opened successfully
#     if not cap.isOpened():
#         print("Error: Could not open video.")
#         exit()

#     # List to store all frames
#     frames = []

#     # Read all frames
#     while True:
#         # Capture frame-by-frame
#         ret, frame = cap.read()
        
#         # If no frame is returned, break the loop
#         if not ret:
#             break

#         # Append the frame to the list
#         frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         frames.append(frame)

#     # Release the video capture object
#     cap.release()
#     return frames

In [3]:
# # Step 3: Function to Extract Embeddings for a List of Frames
# def extract_video_embedding(image_list):
#     """Extracts and averages embeddings for a list of frames."""
#     embeddings = []
#     with torch.no_grad():
#         for image in image_list:
#             input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
#             features = model(input_tensor)
#             embeddings.append(features.squeeze().cpu().numpy())
#     return embeddings


# Step 3: Function to Extract Embeddings for a List of Frames

def extract_video_embedding(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        input_tensor = torch.stack(image_list).to(device)
        features = model(input_tensor)
        features = features.cpu().numpy()
        embeddings = [features[i, :] for i in range(features.shape[0])]

    return embeddings

## Process ##

In [ ]:
# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256" 
video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
process_count = 0
for label_folder in sorted(os.listdir(video_folder)):
    process_count += 1

    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in sorted(os.listdir(full_label_folder)):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        image_list = []
        for image_file in sorted(os.listdir(full_sample_folder)):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            image = transform(image)
            image_list.append(image)
        video_embedding = extract_video_embedding(image_list)
        video_embeddings.append(video_embedding)
        video_labels.append(label)
    if(process_count % 100 == 0):
        print("SAVED @@@@@@@@@@@")
        with open('features_left_hand_frames_small_finetuned_saved.pickle', 'wb') as handle:
            pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('features_left_hand_frames_small_finetuned.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)
abc = 4

## Evaluation ##

In [6]:
with open('features_left_hand_frames_288_to_553.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [5]:
pickle_file11 = open('features_left_hand_frames_553_to_final.pickle', 'rb')
features11,labels11 = pickle.load(pickle_file11)
cc = 5

In [ ]:
## Take average and 

pickle_file = open('/media/osero/SamsungSSD/pickles/features_left_hand_frames_288.pickle', 'rb')
features,labels = pickle.load(pickle_file)
average_features = []
frame_frequency = 5

for feature in features:
    sampled_features = feature[0::frame_frequency]
    average_features.append(np.mean(sampled_features, axis=0))
    ccc = 5

average_features = average_features[0:289]
labels = labels[0:289]
ccc = 5

In [ ]:
# Step 5: Train a Classifier on the Video Embeddings
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(average_features, labels, test_size=0.2)

# Initialize a k-NN classifier
knn = KNeighborsClassifier(n_neighbors = 1)

# Train the classifier
knn.fit(X_train, y_train)

# Predict on the test set and calculate accuracy
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

## Linear Classifier (MLP) ##

In [20]:
from torch.utils.data import DataLoader, Dataset

# Define a Dataset class for loading video features and labels
class VideoDataset(Dataset):
    def __init__(self, features, labels):
        self.labels = labels
        self.features = features
            
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx]).float(), torch.tensor(self.labels[idx]).long()


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
# import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Define an MLP classifier
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLPClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [11]:
# Split features into training and testing sets
train_features, test_features, train_labels, test_labels  = train_test_split(average_features, labels, test_size=0.2)

# Create Dataset and DataLoader
train_dataset = VideoDataset(train_features, train_labels)
test_dataset = VideoDataset(test_features, test_labels)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# Define MLP classifier, loss, and optimizer, and move to GPU
input_dim = train_dataset[0][0].size(0)  # Get input dimension from a single feature
num_classes = len(set(labels))
mlp_classifier = MLPClassifier(input_dim=input_dim, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_classifier.parameters(), lr=0.001)
num_epochs = 10

In [ ]:
# Training loop with batching
for epoch in range(num_epochs):
    mlp_classifier.train()
    running_loss = 0.0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        optimizer.zero_grad()
        outputs = mlp_classifier(batch_features)
        
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [ ]:
# Training loop
for epoch in range(num_epochs):
    mlp_classifier.train()
    optimizer.zero_grad()
    outputs = mlp_classifier(train_features_tensor)
    loss = criterion(outputs, train_labels_tensor)
    loss.backward()
    optimizer.step()
    
print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# for idx, (features, labels) in enumerate(loop):
#     features = features.to(device)
#     labels = labels.to(device)

#     optimizer.zero_grad()
#     outputs = model(features)
#     loss = criterion(outputs, labels)

#     predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
#     correct = (predictions == labels).sum().item()
#     accuracy = correct / batch_size

#     loss.backward()
#     optimizer.step()

#     loop.set_description(f"Epoch [{epoch}/{num_epochs}]")
#     loop.set_postfix(loss=loss.item(), acc=accuracy)

In [ ]:
# Evaluation
mlp_classifier.eval()
with torch.no_grad():
    test_outputs = mlp_classifier(test_features_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = accuracy_score(test_labels_tensor.cpu().numpy(), predicted.cpu().numpy())
    print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("PyTorch Version:", torch.__version__)

## https://github.com/purnasai/Dino_V2/blob/main/3.DinoV2_VS_ResnetClassification.ipynb ##

In [5]:
# Split features into training and testing sets
train_features, test_features, train_labels, test_labels  = train_test_split(average_features, labels, test_size=0.2)

# Create Dataset and DataLoader
train_dataset = VideoDataset(train_features, train_labels)
test_dataset = VideoDataset(test_features, test_labels)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# Define MLP classifier, loss, and optimizer, and move to GPU
input_dim = train_dataset[0][0].size(0)  # Get input dimension from a single feature
num_classes = len(set(labels))
mlp_classifier = MLPClassifier(input_dim=input_dim, num_classes=num_classes).to(device)


criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
optimizer = optim.Adam(mlp_classifier.parameters(), lr=0.000001)
     

In [ ]:
for epoch in range(6):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = mlp_classifier(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 50:.3f}')
            running_loss = 0.0

print('Finished Training')